In [ ]:
def main(datasource, start_date, end_date):
    """
    多因子融合策略 - Multi-Factor Fusion Strategy
    
    基于以下4个子因子的融合：
    1. 订单簿压力因子 (35%) - Order Book Pressure
    2. 量价协同因子 (30%) - Price-Volume Synergy 
    3. 短期反转因子 (25%) - Short-term Mean Reversion
    4. 市场微观结构因子 (10%) - Market Microstructure
    
    Args:
        datasource (str): Datasource table name
        start_date (str): Start date in 'YYYY-MM-DD HH:MM:SS' format
        end_date (str): End date in 'YYYY-MM-DD HH:MM:SS' format

    Returns:
        pd.DataFrame: Factor data with columns ['date', 'instrument', 'factor']
    """
    import pandas as pd
    import dai

    sql = f"""
    WITH cte_snapshot AS (
        SELECT
            date, 
            instrument_id, 
            price, 
            volume,
            amount,
            
            -- 交易日和时间分段
            strftime(date, '%Y-%m-%d') as trading_day,
            
            -- 计算中间价格
            (ask_price1 + bid_price1) / 2.0 as mid_price,
            
            -- ============ 因子1：订单簿压力 (简化版，使用3档) ============
            -- 加权3档买方量 (降低数据依赖)
            (
                COALESCE(bid_volume1, 0) * 1.0 + 
                COALESCE(bid_volume2, 0) * 0.7 + 
                COALESCE(bid_volume3, 0) * 0.5
            ) as weight_bid_3,
            
            -- 加权3档卖方量
            (
                COALESCE(ask_volume1, 0) * 1.0 + 
                COALESCE(ask_volume2, 0) * 0.7 + 
                COALESCE(ask_volume3, 0) * 0.5
            ) as weight_ask_3,
            
            -- 订单簿不平衡度
            (weight_bid_3 - weight_ask_3) / (weight_bid_3 + weight_ask_3 + 1e-8) as order_imbalance,
            
            -- 价差因子
            (ask_price1 - bid_price1) / ((ask_price1 + bid_price1) / 2.0 + 1e-8) as relative_spread,
            
            -- 原始订单簿压力
            order_imbalance / (sqrt(abs(relative_spread)) + 1e-8) as raw_book_pressure,
            
            -- ============ 因子2：量价协同 ============
            -- 成交量变化率
            COALESCE(volume, 0) as curr_volume,
            
            -- ============ 因子3 & 4：需要收益率和微观结构 ============
            -- 买卖价差的绝对值（微观结构信号）
            abs(ask_price1 - bid_price1) as bid_ask_spread,
            
            -- 15分钟时间分段
            CASE 
                -- 上午时间段
                WHEN strftime(date, '%H%M') >= '0930' AND strftime(date, '%H%M') < '0945' THEN 94500
                WHEN strftime(date, '%H%M') >= '0945' AND strftime(date, '%H%M') < '1000' THEN 100000
                WHEN strftime(date, '%H%M') >= '1000' AND strftime(date, '%H%M') < '1015' THEN 101500
                WHEN strftime(date, '%H%M') >= '1015' AND strftime(date, '%H%M') < '1030' THEN 103000
                WHEN strftime(date, '%H%M') >= '1030' AND strftime(date, '%H%M') < '1045' THEN 104500
                WHEN strftime(date, '%H%M') >= '1045' AND strftime(date, '%H%M') < '1100' THEN 110000
                WHEN strftime(date, '%H%M') >= '1100' AND strftime(date, '%H%M') < '1115' THEN 111500
                WHEN strftime(date, '%H%M') >= '1115' AND strftime(date, '%H%M') <= '1130' THEN 113000
                -- 下午时间段
                WHEN strftime(date, '%H%M') >= '1300' AND strftime(date, '%H%M') < '1315' THEN 131500
                WHEN strftime(date, '%H%M') >= '1315' AND strftime(date, '%H%M') < '1330' THEN 133000
                WHEN strftime(date, '%H%M') >= '1330' AND strftime(date, '%H%M') < '1345' THEN 134500
                WHEN strftime(date, '%H%M') >= '1345' AND strftime(date, '%H%M') < '1400' THEN 140000
                WHEN strftime(date, '%H%M') >= '1400' AND strftime(date, '%H%M') < '1415' THEN 141500
                WHEN strftime(date, '%H%M') >= '1415' AND strftime(date, '%H%M') < '1430' THEN 143000
                WHEN strftime(date, '%H%M') >= '1430' AND strftime(date, '%H%M') < '1445' THEN 144500
                WHEN strftime(date, '%H%M') >= '1445' AND strftime(date, '%H%M') < '1457' THEN 150000
                ELSE -1
            END as time_segment
            
        FROM {datasource}
        WHERE time_segment != -1
    ),
    
    -- 计算滚动特征
    cte_rolling AS (
        SELECT 
            *,
            -- 价格序列（用于动量计算）
            lag(mid_price, 1) OVER (PARTITION BY instrument_id, trading_day, time_segment ORDER BY date) as prev_mid_price,
            lag(curr_volume, 1) OVER (PARTITION BY instrument_id, trading_day, time_segment ORDER BY date) as prev_volume,
            
            -- 收益率
            CASE 
                WHEN prev_mid_price IS NOT NULL AND prev_mid_price > 0
                THEN (mid_price - prev_mid_price) / prev_mid_price
                ELSE 0
            END as returns,
            
            -- 成交量变化率
            CASE
                WHEN prev_volume IS NOT NULL AND prev_volume > 0
                THEN (curr_volume - prev_volume) / prev_volume
                ELSE 0
            END as volume_change
            
        FROM cte_snapshot
    ),
    
    -- 计算窗口统计量和子因子
    cte_subfactors AS (
        SELECT
            trading_day, 
            time_segment, 
            instrument_id,
            
            -- ============ 因子1: 订单簿压力 (35%) ============
            -- 波动率感知的订单簿压力
            avg(raw_book_pressure) as book_pressure_mean,
            nanstd(raw_book_pressure) as book_pressure_std,
            
            CASE
                WHEN book_pressure_std > 1e-6
                THEN (last(raw_book_pressure) - book_pressure_mean) / book_pressure_std
                ELSE 0
            END as book_pressure_zscore,
            
            -- 波动率阈值
            nanstd(returns) as volatility,
            CASE
                WHEN COUNT(*) > 5
                THEN quantile(abs(returns), 0.75)
                ELSE 0.01
            END as vol_threshold,
            
            -- 波动率调整后的订单簿压力因子
            CASE
                WHEN volatility > vol_threshold
                THEN book_pressure_zscore * 0.6  -- 高波动时降低权重
                ELSE book_pressure_zscore
            END as factor1_book_pressure,
            
            -- ============ 因子2: 量价协同 (30%) ============
            -- 价格动量
            avg(returns) as avg_returns,
            
            -- 成交量动量
            avg(volume_change) as avg_volume_change,
            
            -- 量价相关性（当样本足够时）
            CASE
                WHEN COUNT(*) >= 5
                THEN corr(returns, volume_change)
                ELSE 0
            END as pv_correlation,
            
            -- 量价协同因子 = 动量 * 量价一致性
            avg_returns * (1 + pv_correlation) / 2.0 as factor2_pv_synergy,
            
            -- ============ 因子3: 短期反转 (25%) ============
            -- 价格偏离均值的程度
            avg(mid_price) as avg_mid_price,
            last(mid_price) as last_mid_price,
            
            CASE
                WHEN avg_mid_price > 0
                THEN (last_mid_price - avg_mid_price) / avg_mid_price
                ELSE 0
            END as price_deviation,
            
            -- 反转因子（负号表示做反）
            -tanh(price_deviation * 8) as factor3_reversal,
            
            -- ============ 因子4: 市场微观结构 (10%) ============
            -- 价差变化（流动性信号）
            avg(relative_spread) as avg_spread,
            last(relative_spread) as last_spread,
            
            -- 价差扩大 -> 流动性恶化 -> 负面信号
            CASE
                WHEN avg_spread > 1e-6
                THEN -(last_spread - avg_spread) / avg_spread
                ELSE 0
            END as spread_change,
            
            -- 成交量不规则性（波动 = 不确定性）
            CASE
                WHEN avg(curr_volume) > 1e-6
                THEN -nanstd(curr_volume) / avg(curr_volume)
                ELSE 0
            END as volume_irregularity,
            
            -- 微观结构因子
            (spread_change + volume_irregularity) / 2.0 as factor4_microstructure
            
        FROM cte_rolling
        GROUP BY instrument_id, trading_day, time_segment
        ORDER BY instrument_id, time_segment
    ),
    
    -- 横截面排名并融合
    cte_ranked AS (
        SELECT
            *,
            -- 对每个子因子进行横截面排名（0-1标准化）
            percent_rank() OVER (PARTITION BY trading_day, time_segment ORDER BY factor1_book_pressure) as rank1,
            percent_rank() OVER (PARTITION BY trading_day, time_segment ORDER BY factor2_pv_synergy) as rank2,
            percent_rank() OVER (PARTITION BY trading_day, time_segment ORDER BY factor3_reversal) as rank3,
            percent_rank() OVER (PARTITION BY trading_day, time_segment ORDER BY factor4_microstructure) as rank4
        FROM cte_subfactors
    ),
    
    -- 加权融合
    cte_final AS (
        SELECT
            trading_day,
            time_segment,
            instrument_id,
            
            -- 多因子融合：排名加权
            -- 权重: 订单簿35%, 量价30%, 反转25%, 微观结构10%
            0.35 * rank1 + 0.30 * rank2 + 0.25 * rank3 + 0.10 * rank4 as composite_score,
            
            -- 横截面标准化
            avg(composite_score) OVER (PARTITION BY trading_day, time_segment) as score_mean,
            nanstd(composite_score) OVER (PARTITION BY trading_day, time_segment) as score_std,
            
            CASE
                WHEN score_std > 1e-6
                THEN (composite_score - score_mean) / score_std
                ELSE 0
            END as final_zscore,
            
            -- 最终因子：tanh映射到[-1, 1]
            tanh(final_zscore) as factor
            
        FROM cte_ranked
    )
    
    -- 映射instrument_id到instrument
    SELECT
        -- 转换为15分钟的date列
        CAST(CONCAT(
            f.trading_day,
            ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.factor as factor
    FROM cte_final f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={'date': [start_date, end_date]}).df()
    return df


if __name__ == '__main__':
    """
    开发和测试代码
    
    使用方法：
    1. 调整日期范围进行测试
    2. 使用factorlens模块评估因子表现
    3. 根据回测结果调整子因子权重（目前：35%, 30%, 25%, 10%）
    
    策略说明：
    - 本策略采用多因子融合，降低单一因子风险
    - 使用排名加权而非直接加权，提高稳健性
    - 波动率自适应调整，在不同市场环境下保持稳定
    - 因子组合覆盖：微观结构、动量、反转、量价关系
    """
    from bigmodule import M
    from datetime import datetime
    import structlog

    logger = structlog.get_logger()

    datasource = 'cpt_dwc_2026_stock_hs300_snapshot'
    start_date = '2023-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"计算多因子融合策略，时间范围: {start_date} 到 {end_date}")
    t1 = datetime.now()
    data = main(datasource, start_date, end_date)
    t2 = datetime.now()
    total_seconds = (t2 - t1).total_seconds()
    minutes = int(total_seconds // 60)
    seconds = int(total_seconds % 60)
    logger.info(f"计算耗时: {minutes}分{seconds}秒")
    logger.info(f"因子数据维度: {data.shape}")
    logger.info(f"\n样本数据:\n{data.head()}")
    logger.info(f"\n因子统计:\n{data['factor'].describe()}")

    # 因子评估
    results = M.eval_dwc._latest(
        data=data
    )
    logger.info("因子评估完成！")